In [227]:
import altair as alt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
df = pd.read_csv('spotify_top_1000_tracks.csv')
#df.head()

,track_name,artist,album,release_date,popularity,spotify_url,id,duration_min
0,All The Stars (with SZA),Kendrick Lamar,Black Panther The Album Music From And Inspire...,2018-02-09,95,https://open.spotify.com/track/3GCdLUSnKSMJhs4...,3GCdLUSnKSMJhs4Tj6CV3s,3.869767
1,Starboy,The Weeknd,Starboy,2016-11-25,90,https://open.spotify.com/track/7MXVkk9YMctZqd1...,7MXVkk9YMctZqd1Srtv4MB,3.840883
2,Señorita,Shawn Mendes,Señorita,2019-06-21,80,https://open.spotify.com/track/0TK2YIli7K1leLo...,0TK2YIli7K1leLovkQiNik,3.182667
3,Heat Waves,Glass Animals,Dreamland,2020-08-07,87,https://open.spotify.com/track/3USxtqRwSYz57Ew...,3USxtqRwSYz57Ewm6wWRMp,3.980083
4,Let Me Love You,DJ Snake,Encore,2016-08-05,87,https://open.spotify.com/track/0lYBSQXN6rCTvUZ...,0lYBSQXN6rCTvUZvg9S0lU,3.432433


In [228]:
#don't need these two columns
df = df.drop(columns =['id','spotify_url'])
#convert release date to year and month
df['release_date'] = pd.to_datetime(df['release_date'])
df['Year'] = df['release_date'].dt.year
df['Month'] = df['release_date'].dt.month
#df.info()
#vis i want to do
#top 5 artists by number of songs, time distribution of those hits
#duration frequency, release year and release month freq
#correlation matrix and lin regression


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   track_name    1000 non-null   object        
 1   artist        1000 non-null   object        
 2   album         1000 non-null   object        
 3   release_date  1000 non-null   datetime64[ns]
 4   popularity    1000 non-null   int64         
 5   duration_min  1000 non-null   float64       
 6   Year          1000 non-null   int64         
 7   Month         1000 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(3)
memory usage: 62.6+ KB


In [229]:

#new df grouped by artist song count
songCount = df
songCount['Songs'] = df['artist'].map(df['artist'].value_counts())
songCount= songCount.sort_values('Songs', ascending=False).drop_duplicates(subset=['artist'])
songCount.head()

#need a df with just the five top artists
names =['The Weeknd','Ariana Grande', 'Alan Walker', 'Avril Lavigne', 'Taylor Swift']
top5Artists = songCount[songCount['artist'].isin(names)]
#top5Artists.info()

#need df with duration and popularity grouped
durationDF =df.drop(columns=['artist','track_name','album','release_date','Year','Month','Songs'])
durationDF = durationDF.sort_values(['duration_min','popularity'], ascending = False)

monthDF =df.drop(columns=['artist','track_name','album','duration_min','Songs'])
#dropping rows with date of 1-1-xxxx because those are placeholder dates 
monthDF = monthDF.drop(monthDF[(monthDF['release_date'].dt.month==1)&(monthDF['release_date'].dt.day==1)].index)
monthDF = monthDF.sort_values(['Month','popularity'],ascending = False)


<class 'pandas.core.frame.DataFrame'>
Int64Index: 5 entries, 690 to 239
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   track_name    5 non-null      object        
 1   artist        5 non-null      object        
 2   album         5 non-null      object        
 3   release_date  5 non-null      datetime64[ns]
 4   popularity    5 non-null      int64         
 5   duration_min  5 non-null      float64       
 6   Year          5 non-null      int64         
 7   Month         5 non-null      int64         
 8   Songs         5 non-null      int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(3)
memory usage: 400.0+ bytes


In [230]:
#monthDF.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 886 entries, 688 to 999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   release_date  886 non-null    datetime64[ns]
 1   popularity    886 non-null    int64         
 2   Year          886 non-null    int64         
 3   Month         886 non-null    int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 34.6 KB


In [231]:
#top 5 artists visualization
top5Bar = alt.Chart(top5Artists).mark_bar().encode(
    alt.X('artist').axis().title('Artist'),
    y='Songs',
    color = alt.Color('artist').scale(scheme='category10')
    
).properties(width =500, height =300)
top5Bar.show()

alt.Chart(...)

In [232]:
monthFreq= alt.Chart(monthDF).mark_bar(size=30).encode(
    alt.X('Month'),
    alt.Y('count()', title='Frequency'),
    color = alt.Color('Month', legend=None).scale(scheme = 'category20'),
    
).properties(width = 500)
monthFreq.show()

alt.Chart(...)

In [233]:
#massive outlier in january so i'm exploring that here
#df.sort_values('release_date').head()
#okay so from this I can tell that any song released before Spotify launched (or it's release date is unknown) is marked as 
#being released on January 1st of that year.
#need to remove that. will just fix it above


,track_name,artist,album,release_date,popularity,duration_min,Year,Month,Songs
517,"Money, Money, Money",ABBA,Arrival,1976-01-01,72,3.107100,1976,1,2
602,Goo Goo Muck,The Cramps,Psychedelic Jungle,1981-01-01,67,3.100633,1981,1,1
694,Smalltown Boy,Bronski Beat,The Age Of Consent,1984-01-01,2,5.055100,1984,1,1
431,Running Up That Hill (A Deal With God),Kate Bush,Hounds Of Love,1985-01-01,11,4.982217,1985,1,1
512,Cheri Cheri Lady,Modern Talking,Let's Talk About Love,1985-01-01,82,3.772217,1985,1,1


In [234]:
#year frequency
yearFreq = alt.Chart(monthDF).mark_bar(size=15).encode(
    alt.X('Year', scale = alt.Scale(domain = [1990,2026]), axis = alt.Axis(format='.0f')),
    alt.Y('count()', title='Frequency'),
    color = alt.Color('Year', legend=None).scale(scheme = 'category20')
).properties(width = 700)
yearFreq.show()

alt.Chart(...)

In [322]:
# interactive graph for popularity and year
pop2024 = df[df['Year']==2024]
scatter = alt.Chart(pop2024).mark_circle().encode(
    alt.X('Month'),
    alt.Y('popularity'),
    color=alt.Color('Month',legend=None).scale(scheme="category10"),
    tooltip = ['track_name','artist','album','release_date']).interactive().properties(title='2024 Month vs Popularity')
scatter.show()

alt.Chart(...)

In [319]:
duraDen= alt.Chart(durationDF).transform_density('duration_min',
    as_=['duration_min', 'Density'],).mark_line().encode(
    alt.X('duration_min:Q'),
    alt.Y('Density:Q')
)
duraDen.show()

alt.Chart(...)

In [238]:
#doing the above but as the bar frequency graph just to see it two different ways
duraFreq = alt.Chart(durationDF).mark_bar(size=25).encode(
    alt.X('duration_min').bin(),
    alt.Y('count()', title='Frequency')
).properties(width = 700)
duraFreq.show()

alt.Chart(...)

In [316]:
#interactive vis for the top 5 artist
click = alt.selection_point(fields=['artist'],encodings=['color'])
top5onlyDF= df[df['artist'].isin(names)]
top5scatter = alt.Chart(top5onlyDF).mark_circle().encode(
    alt.X('Year',scale = alt.Scale(domain = [2000,2026]),axis = alt.Axis(format='.0f')),
    alt.Y('popularity'),
    color=alt.Color('artist',legend=None).scale(scheme="category10"),
    tooltip = ['track_name','artist','album','release_date'],
    opacity=alt.when(click).then(alt.value(1)).otherwise(alt.value(0.2))
    ).transform_filter(
        click
    ).add_params(click).properties(title="Top 5 Artists' Hits").interactive()
legend = alt.Chart(top5onlyDF).mark_rect().encode(
    y=alt.Y('artist', axis=alt.Axis(title='Select Artist')),
    color = alt.condition(click, 'artist',
                         alt.value('lightgray'),legend=None),
    size=alt.value(250)
).add_params(
    click
).properties(title= 'Legend')
chart = (top5scatter|legend)
chart

alt.HConcatChart(...)

In [ ]:
#okay now just put them all together in a vis
#write conclusions